# 02 — NLI-Entailment Metric

Splits each `response` into sentences (hypotheses), checks each against `source_info` (premise) using an off-the-shelf NLI model (`roberta-large-mnli`), and aggregates per-response as the fraction of response sentences classified as entailment. Same canonical split (`ragtruth_split_assignment.csv`), same threshold/bootstrap procedure as ROUGE/BERTScore, so results stay directly comparable.

Run after `00_setup_and_data_exploration.ipynb` and `01_baseline_metrics.ipynb`. **Needs GPU** (Runtime → Change runtime type → T4 GPU).

In [ ]:
!pip install -q transformers sentencepiece

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU — switch Runtime > Change runtime type > T4 GPU before continuing.')

CUDA available: True
GPU: Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -q nltk

import pandas as pd
import ast
import nltk
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize

DRIVE_DIR = '/content/drive/MyDrive/rag-eval-metrics'

eval_df = pd.read_csv(f'{DRIVE_DIR}/ragtruth_eval_dataframe.csv')
eval_df['labels'] = eval_df['labels'].apply(lambda x: [] if pd.isna(x) or x == '' else ast.literal_eval(x))

split_df = pd.read_csv(f'{DRIVE_DIR}/ragtruth_split_assignment.csv')

print('eval_df:', eval_df.shape)
assert eval_df.shape[0] == 1781, f"Unexpected row count: {eval_df.shape[0]}"

eval_df['response_sentences'] = eval_df['response'].apply(sent_tokenize)
eval_df['n_sentences'] = eval_df['response_sentences'].apply(len)

print('\nExample split (first row):')
print(eval_df.iloc[0]['response_sentences'])

print('\nSentences per response — describe:')
print(eval_df['n_sentences'].describe())

n_zero = (eval_df['n_sentences'] == 0).sum()
print(f'\nResponses with 0 sentences after splitting: {n_zero}')

eval_df: (1781, 11)

Example split (first row):
['"The Mex Authentic" is a popular Mexican restaurant located at 413 State St, Santa Barbara, CA.', 'Known for its outstanding food and service, the restaurant has a high average rating of 4.5 stars.', 'The restaurant offers specialty food, with a strong focus on ethnic Mexican cuisine, and also features a bar.', 'It is noted for being a good location for group gatherings.', 'The restaurant does not offer reservations, outdoor seating, WiFi or music, and its ambience is described as casual.', 'Available parking is limited to street parking.', 'The restaurant is open from 11:30 to 21:00 or 22:00 on Thursday through Saturday, and from 11:00 to 15:00 on Sunday.', "Reviewers have praised the restaurant's excellent service and food, with one reviewer mentioning the server Giovanni by name for his great recommendations."]

Sentences per response — describe:
count    1781.000000
mean        7.606962
std         4.351215
min         1.000000
25% 

In [ ]:
#Attach and verify the canonical split
# Attach the existing canonical split assignment by response ID.
# Do not create a new train/validation/test split.

if 'split' not in eval_df.columns:
    assert split_df['id'].is_unique, "Duplicate IDs found in split assignment."
    assert eval_df['id'].is_unique, "Duplicate IDs found in evaluation dataframe."

    eval_df = eval_df.merge(
        split_df[['id', 'split']],
        on='id',
        how='left',
        validate='one_to_one'
    )

# Verify every response has a split assignment.
assert eval_df['split'].notna().all(), "Some responses are missing split assignments."
assert len(eval_df) == 1781, f"Unexpected row count: {len(eval_df)}"

print("Dataset shape:", eval_df.shape)
print("\nSplit counts:")
print(eval_df['split'].value_counts())

print("\nHallucination rate by split:")
print(
    eval_df.groupby('split')['has_hallucination']
    .mean()
    .round(4)
)

print("\nTask distribution by split:")
print(pd.crosstab(eval_df['split'], eval_df['task_type']))

Dataset shape: (1781, 13)

Split counts:
split
train    1528
test      253
Name: count, dtype: int64

Hallucination rate by split:
split
test     0.3874
train    0.4359
Name: has_hallucination, dtype: float64

Task distribution by split:
task_type  Data2txt   QA  Summary
split                            
test             94   84       75
train           533  499      496


In [ ]:
print("Split assignment shape:", split_df.shape)
print("\nColumns:")
print(split_df.columns.tolist())

print("\nSplit counts in the original assignment file:")
print(split_df['split'].value_counts(dropna=False))

print("\nDuplicate IDs in split file:", split_df['id'].duplicated().sum())
print("Duplicate IDs in eval_df:", eval_df['id'].duplicated().sum())

print("\nFirst rows of split assignment:")
display(split_df.head())

print("\nUnique split labels:")
print(sorted(split_df['split'].dropna().unique()))

Split assignment shape: (1781, 2)

Columns:
['id', 'split']

Split counts in the original assignment file:
split
train    1068
test      357
val       356
Name: count, dtype: int64

Duplicate IDs in split file: 0
Duplicate IDs in eval_df: 0

First rows of split assignment:


,id,split
0,11538,train
1,10679,train
2,7264,test
3,9643,test
4,6832,train



Unique split labels:
['test', 'train', 'val']


In [ ]:
#Refresh the split from the canonical file

# Remove any existing split column and reattach the canonical assignments.
eval_df = eval_df.drop(columns=['split'], errors='ignore')

eval_df = eval_df.merge(
    split_df[['id', 'split']],
    on='id',
    how='left',
    validate='one_to_one'
)

# Verify the merge.
assert eval_df['split'].notna().all(), "Some rows are missing split assignments."
assert len(eval_df) == 1781, f"Unexpected row count: {len(eval_df)}"

print("Dataset shape:", eval_df.shape)

print("\nCanonical split counts:")
print(eval_df['split'].value_counts())

print("\nHallucination rate by split:")
print(
    eval_df.groupby('split')['has_hallucination']
    .mean()
    .round(4)
)

print("\nTask distribution by split:")
print(pd.crosstab(eval_df['split'], eval_df['task_type']))

Dataset shape: (1781, 13)

Canonical split counts:
split
train    1068
test      357
val       356
Name: count, dtype: int64

Hallucination rate by split:
split
test     0.4286
train    0.4288
val      0.4298
Name: has_hallucination, dtype: float64

Task distribution by split:
task_type  Data2txt   QA  Summary
split                            
test            126  116      115
train           376  350      342
val             125  117      114


In [ ]:
#Load the NLI Model
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MODEL_NAME = "roberta-large-mnli"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Loading model:", MODEL_NAME)
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
nli_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

nli_model.to(device)
nli_model.eval()

print("\nModel loaded successfully.")
print("Device:", next(nli_model.parameters()).device)
print("Label mapping:", nli_model.config.id2label)

Loading model: roberta-large-mnli
Device: cuda


config.json:   0%|          | 0.00/688 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.43GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-large-mnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Model loaded successfully.
Device: cuda:0
Label mapping: {0: 'CONTRADICTION', 1: 'NEUTRAL', 2: 'ENTAILMENT'}


In [ ]:
#Check the sentence-splitting columns
print("eval_df columns:")
print(eval_df.columns.tolist())

print("\nColumns containing 'sent' or 'response':")
print([
    col for col in eval_df.columns
    if "sent" in col.lower() or "response" in col.lower()
])

print("\nData types:")
print(eval_df.dtypes)

eval_df columns:
['id', 'source_id', 'model', 'temperature', 'labels', 'quality', 'response', 'task_type', 'source_info', 'has_hallucination', 'response_sentences', 'n_sentences', 'split']

Columns containing 'sent' or 'response':
['response', 'response_sentences', 'n_sentences']

Data types:
id                      int64
source_id               int64
model                  object
temperature           float64
labels                 object
quality                object
response               object
task_type              object
source_info            object
has_hallucination        bool
response_sentences     object
n_sentences             int64
split                  object
dtype: object


In [ ]:
# Prepare source-context / response-sentence pairs for NLI scoring.

nli_pairs = []

for _, row in eval_df.iterrows():
    response_id = row["id"]
    premise = str(row["source_info"])
    sentences = row["response_sentences"]

    for sentence_index, hypothesis in enumerate(sentences):
        nli_pairs.append({
            "id": response_id,
            "sentence_index": sentence_index,
            "premise": premise,
            "hypothesis": hypothesis
        })

nli_pairs_df = pd.DataFrame(nli_pairs)

print("Number of responses:", len(eval_df))
print("Number of response sentences:", len(nli_pairs_df))

print("\nPrepared pair columns:")
print(nli_pairs_df.columns.tolist())

print("\nExample NLI pair:")
display(nli_pairs_df[["id", "sentence_index", "premise", "hypothesis"]].head(1))

Number of responses: 1781
Number of response sentences: 13548

Prepared pair columns:
['id', 'sentence_index', 'premise', 'hypothesis']

Example NLI pair:


,id,sentence_index,premise,hypothesis
0,11538,0,"{'name': 'The Mex Authentic', 'address': '413 ...","""The Mex Authentic"" is a popular Mexican resta..."


In [ ]:
#Inspect source_info
# Inspect the structure of source_info before converting it into NLI premise text.

sample_source = eval_df["source_info"].iloc[0]

print("Type:", type(sample_source))
print("\nSample source_info:")
print(sample_source)

print("\nKeys (if dictionary):")
if isinstance(sample_source, dict):
    print(list(sample_source.keys()))

Type: <class 'str'>

Sample source_info:
{'name': 'The Mex Authentic', 'address': '413 State St', 'city': 'Santa Barbara', 'state': 'CA', 'categories': 'Food, Specialty Food, Nightlife, Restaurants, Local Flavor, Ethnic Food, Beer, Wine & Spirits, Bars, Mexican', 'hours': {'Thursday': '11:30-21:0', 'Friday': '11:30-22:0', 'Saturday': '11:30-22:0', 'Sunday': '11:0-15:0'}, 'attributes': {'BusinessParking': {'garage': False, 'street': True, 'validated': False, 'lot': False, 'valet': False}, 'RestaurantsReservations': False, 'OutdoorSeating': False, 'WiFi': 'no', 'RestaurantsTakeOut': True, 'RestaurantsGoodForGroups': True, 'Music': False, 'Ambience': {'romantic': False, 'intimate': False, 'classy': False, 'hipster': False, 'divey': False, 'touristy': False, 'trendy': False, 'upscale': False, 'casual': True}}, 'business_stars': 4.5, 'review_info': [{'review_stars': 5.0, 'review_date': '2018-03-04 04:48:58', 'review_text': 'What happened!?!? I moved back to Ohio and they close? It was some 

In [ ]:
#Convert source_info into readable premise text
import ast

def flatten_source_info(value):
    """Convert nested source information into readable text."""
    parts = []

    if isinstance(value, dict):
        for key, item in value.items():
            parts.append(f"{key}: {flatten_source_info(item)}")

    elif isinstance(value, list):
        for item in value:
            parts.append(flatten_source_info(item))

    elif value is not None:
        parts.append(str(value))

    return ". ".join(part for part in parts if part)


def source_to_text(source):
    """Parse source_info strings and convert them to readable text."""
    if isinstance(source, str):
        try:
            parsed_source = ast.literal_eval(source)
        except (ValueError, SyntaxError):
            return source
    else:
        parsed_source = source

    return flatten_source_info(parsed_source)


# Create readable source-context text for NLI.
eval_df["source_context_text"] = eval_df["source_info"].apply(source_to_text)

print("Original source_info type:", type(eval_df["source_info"].iloc[0]))
print("Converted source context type:", type(eval_df["source_context_text"].iloc[0]))

print("\nExample converted premise:")
print(eval_df["source_context_text"].iloc[0][:1500])

print("\nMissing/empty converted contexts:",
      eval_df["source_context_text"].isna().sum() +
      (eval_df["source_context_text"].str.strip() == "").sum())

Original source_info type: <class 'str'>
Converted source context type: <class 'str'>

Example converted premise:
name: The Mex Authentic. address: 413 State St. city: Santa Barbara. state: CA. categories: Food, Specialty Food, Nightlife, Restaurants, Local Flavor, Ethnic Food, Beer, Wine & Spirits, Bars, Mexican. hours: Thursday: 11:30-21:0. Friday: 11:30-22:0. Saturday: 11:30-22:0. Sunday: 11:0-15:0. attributes: BusinessParking: garage: False. street: True. validated: False. lot: False. valet: False. RestaurantsReservations: False. OutdoorSeating: False. WiFi: no. RestaurantsTakeOut: True. RestaurantsGoodForGroups: True. Music: False. Ambience: romantic: False. intimate: False. classy: False. hipster: False. divey: False. touristy: False. trendy: False. upscale: False. casual: True. business_stars: 4.5. review_info: review_stars: 5.0. review_date: 2018-03-04 04:48:58. review_text: What happened!?!? I moved back to Ohio and they close? It was some of the best Mexican fare in town and 

In [ ]:
#Run NLI inference in batches
import torch
import pandas as pd
from tqdm.auto import tqdm

# Rebuild sentence-level pairs using the cleaned source context.
nli_pairs = []

for _, row in eval_df.iterrows():
    for sentence_index, hypothesis in enumerate(row["response_sentences"]):
        nli_pairs.append({
            "id": row["id"],
            "sentence_index": sentence_index,
            "premise": row["source_context_text"],
            "hypothesis": hypothesis
        })

nli_pairs_df = pd.DataFrame(nli_pairs)

# Confirm the expected number of sentence pairs.
print("Sentence pairs to score:", len(nli_pairs_df))

# NLI inference settings.
BATCH_SIZE = 16
MAX_LENGTH = 512

all_probabilities = []
all_predictions = []

nli_model.eval()

for start in tqdm(range(0, len(nli_pairs_df), BATCH_SIZE)):
    batch = nli_pairs_df.iloc[start:start + BATCH_SIZE]

    encoded = tokenizer(
        batch["premise"].tolist(),
        batch["hypothesis"].tolist(),
        truncation="only_first",
        max_length=MAX_LENGTH,
        padding=True,
        return_tensors="pt"
    )

    encoded = {key: value.to(device) for key, value in encoded.items()}

    with torch.no_grad():
        outputs = nli_model(**encoded)
        probabilities = torch.softmax(outputs.logits, dim=1)

    all_probabilities.extend(probabilities.cpu().tolist())
    all_predictions.extend(torch.argmax(probabilities, dim=1).cpu().tolist())

# Attach the model outputs to the sentence-pair dataframe.
nli_pairs_df["contradiction_probability"] = [
    probs[0] for probs in all_probabilities
]
nli_pairs_df["neutral_probability"] = [
    probs[1] for probs in all_probabilities
]
nli_pairs_df["entailment_probability"] = [
    probs[2] for probs in all_probabilities
]
nli_pairs_df["predicted_label_id"] = all_predictions
nli_pairs_df["predicted_label"] = nli_pairs_df["predicted_label_id"].map(
    nli_model.config.id2label
)

print("\nNLI inference completed.")
print("Scored sentence pairs:", len(nli_pairs_df))

print("\nPredicted label counts:")
print(nli_pairs_df["predicted_label"].value_counts())

print("\nSample scored pairs:")
display(
    nli_pairs_df[
        [
            "id",
            "sentence_index",
            "predicted_label",
            "entailment_probability",
            "neutral_probability",
            "contradiction_probability"
        ]
    ].head()
)

Sentence pairs to score: 13548


  0%|          | 0/847 [00:00<?, ?it/s]


NLI inference completed.
Scored sentence pairs: 13548

Predicted label counts:
predicted_label
NEUTRAL          9308
CONTRADICTION    3024
ENTAILMENT       1216
Name: count, dtype: int64

Sample scored pairs:


,id,sentence_index,predicted_label,entailment_probability,neutral_probability,contradiction_probability
0,11538,0,NEUTRAL,0.134717,0.672454,0.192829
1,11538,1,NEUTRAL,0.312292,0.633572,0.054136
2,11538,2,NEUTRAL,0.140516,0.723362,0.136122
3,11538,3,NEUTRAL,0.092889,0.822697,0.084414
4,11538,4,NEUTRAL,0.248897,0.487969,0.263134


Aggregate sentence-level NLI scores by response :-
We’ll calculate the average probability of entailment, neutrality, and contradiction for each response. We’ll also calculate the fraction of sentences predicted as entailment, matching the aggregation approach in the NLI

In [ ]:
# Aggregate sentence-level NLI results into one row per response.

nli_response_scores = (
    nli_pairs_df
    .groupby("id")
    .agg(
        n_sentences=("sentence_index", "count"),
        nli_entailment_mean=("entailment_probability", "mean"),
        nli_neutral_mean=("neutral_probability", "mean"),
        nli_contradiction_mean=("contradiction_probability", "mean"),
        entailment_fraction=("predicted_label", lambda x: (x == "ENTAILMENT").mean()),
        neutral_fraction=("predicted_label", lambda x: (x == "NEUTRAL").mean()),
        contradiction_fraction=("predicted_label", lambda x: (x == "CONTRADICTION").mean())
    )
    .reset_index()
)

print("Response-level NLI scores:", nli_response_scores.shape)

print("\nScore summary:")
display(
    nli_response_scores[
        [
            "nli_entailment_mean",
            "nli_neutral_mean",
            "nli_contradiction_mean",
            "entailment_fraction",
            "neutral_fraction",
            "contradiction_fraction"
        ]
    ].describe().round(4)
)

print("\nSample aggregated scores:")
display(nli_response_scores.head())

Response-level NLI scores: (1781, 8)

Score summary:


,nli_entailment_mean,nli_neutral_mean,nli_contradiction_mean,entailment_fraction,neutral_fraction,contradiction_fraction
count,1781.0000,1781.0000,1781.0000,1781.0000,1781.0000,1781.0000
mean,0.1546,0.6271,0.2183,0.1032,0.7044,0.1924
std,0.1593,0.2491,0.2110,0.2122,0.3607,0.3205
min,0.0023,0.0097,0.0011,0.0000,0.0000,0.0000
25%,0.0364,0.4324,0.0440,0.0000,0.4500,0.0000
50%,0.1099,0.6495,0.1476,0.0000,0.8750,0.0000
75%,0.2221,0.8541,0.3362,0.1250,1.0000,0.2727
max,0.9883,0.9946,0.9357,1.0000,1.0000,1.0000



Sample aggregated scores:


,id,n_sentences,nli_entailment_mean,nli_neutral_mean,nli_contradiction_mean,entailment_fraction,neutral_fraction,contradiction_fraction
0,8,5,0.265096,0.698564,0.036340,0.0,1.0,0.0
1,15,4,0.031334,0.942865,0.025801,0.0,1.0,0.0
2,23,8,0.011838,0.943602,0.044559,0.0,1.0,0.0
3,29,5,0.024450,0.353590,0.621960,0.0,0.2,0.8
4,33,5,0.004784,0.991825,0.003391,0.0,1.0,0.0


In [ ]:
#Merge NLI scores with labels and split

# Keep the ground-truth labels and canonical split for evaluation.
nli_eval_df = eval_df[
    ["id", "task_type", "has_hallucination", "split"]
].merge(
    nli_response_scores,
    on="id",
    how="inner",
    validate="one_to_one"
)

# Verify that every response is represented exactly once.
assert len(nli_eval_df) == len(eval_df) == 1781
assert nli_eval_df["has_hallucination"].notna().all()
assert nli_eval_df["split"].notna().all()

print("NLI evaluation dataframe shape:", nli_eval_df.shape)

print("\nSplit counts:")
print(nli_eval_df["split"].value_counts())

print("\nHallucination label counts:")
print(nli_eval_df["has_hallucination"].value_counts())

print("\nMean entailment fraction by ground-truth label:")
print(
    nli_eval_df.groupby("has_hallucination")["entailment_fraction"]
    .agg(["count", "mean", "std"])
    .round(4)
)

print("\nMean contradiction fraction by ground-truth label:")
print(
    nli_eval_df.groupby("has_hallucination")["contradiction_fraction"]
    .agg(["count", "mean", "std"])
    .round(4)
)

NLI evaluation dataframe shape: (1781, 11)

Split counts:
split
train    1068
test      357
val       356
Name: count, dtype: int64

Hallucination label counts:
has_hallucination
False    1017
True      764
Name: count, dtype: int64

Mean entailment fraction by ground-truth label:
                   count    mean     std
has_hallucination                       
False               1017  0.1194  0.2422
True                 764  0.0816  0.1616

Mean contradiction fraction by ground-truth label:
                   count    mean     std
has_hallucination                       
False               1017  0.2071  0.3388
True                 764  0.1729  0.2934


Create the NLI hallucination signal :-

Since lower entailment is associated with hallucination in our results, we’ll define the NLI hallucination signal as:

{NLI hallucination signal} = 1 - \text{entailment fraction}

A higher value means a smaller fraction of response sentences were classified as entailed by the source context. We’ll select the classification threshold using validation data only, then evaluate once on the held-out test set.

In [ ]:
import numpy as np
from sklearn.metrics import (
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

# Higher score indicates weaker entailment support.
nli_eval_df["nli_hallucination_score"] = (
    1 - nli_eval_df["entailment_fraction"]
)

# Separate the canonical validation and test sets.
val_df = nli_eval_df[nli_eval_df["split"] == "val"].copy()
test_df = nli_eval_df[nli_eval_df["split"] == "test"].copy()

y_val = val_df["has_hallucination"].astype(int)
val_scores = val_df["nli_hallucination_score"]

y_test = test_df["has_hallucination"].astype(int)
test_scores = test_df["nli_hallucination_score"]

# Select the threshold that maximizes validation F1.
precision_vals, recall_vals, thresholds = precision_recall_curve(
    y_val, val_scores
)

f1_vals = (
    2 * precision_vals[:-1] * recall_vals[:-1]
    / (precision_vals[:-1] + recall_vals[:-1] + 1e-12)
)

best_idx = np.argmax(f1_vals)
best_threshold = thresholds[best_idx]
best_val_f1 = f1_vals[best_idx]

print("Validation responses:", len(val_df))
print("Test responses:", len(test_df))
print(f"Selected validation threshold: {best_threshold:.6f}")
print(f"Best validation F1: {best_val_f1:.4f}")

Validation responses: 356
Test responses: 357
Selected validation threshold: 0.600000
Best validation F1: 0.6128


In [ ]:
#Evaluate NLI on the test set

# Apply the validation-selected threshold to the test set.
y_test_pred = (test_scores >= best_threshold).astype(int)

# Calculate test metrics.
test_accuracy = accuracy_score(y_test, y_test_pred)
test_precision = precision_score(y_test, y_test_pred, zero_division=0)
test_recall = recall_score(y_test, y_test_pred, zero_division=0)
test_f1 = f1_score(y_test, y_test_pred, zero_division=0)
test_roc_auc = roc_auc_score(y_test, test_scores)
test_pr_auc = average_precision_score(y_test, test_scores)

cm = confusion_matrix(y_test, y_test_pred)

print("NLI Test Set Results")
print("--------------------")
print(f"Threshold:  {best_threshold:.4f}")
print(f"Accuracy:   {test_accuracy:.4f}")
print(f"Precision:  {test_precision:.4f}")
print(f"Recall:     {test_recall:.4f}")
print(f"F1 Score:   {test_f1:.4f}")
print(f"ROC-AUC:    {test_roc_auc:.4f}")
print(f"PR-AUC:     {test_pr_auc:.4f}")

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_test_pred,
    target_names=["Non-hallucinated", "Hallucinated"],
    zero_division=0
))

NLI Test Set Results
--------------------
Threshold:  0.6000
Accuracy:   0.4762
Precision:  0.4472
Recall:     0.9412
F1 Score:   0.6063
ROC-AUC:    0.5115
PR-AUC:     0.4313

Confusion Matrix:
[[ 26 178]
 [  9 144]]

Classification Report:
                  precision    recall  f1-score   support

Non-hallucinated       0.74      0.13      0.22       204
    Hallucinated       0.45      0.94      0.61       153

        accuracy                           0.48       357
       macro avg       0.60      0.53      0.41       357
    weighted avg       0.62      0.48      0.38       357



In [ ]:
#Bootstrap confidence intervals for NLI - validation-selected threshold fixed at 0.60 throughout.

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# Test labels and continuous NLI scores as NumPy arrays.
y_true = y_test.to_numpy().astype(int)
scores = test_scores.to_numpy()
y_pred = (scores >= best_threshold).astype(int)

N_BOOTSTRAP = 2000
RANDOM_STATE = 42

rng = np.random.default_rng(RANDOM_STATE)

metric_names = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "pr_auc"
]

bootstrap_results = {name: [] for name in metric_names}

for _ in range(N_BOOTSTRAP):
    # Sample test rows with replacement.
    indices = rng.choice(len(y_true), size=len(y_true), replace=True)

    y_b = y_true[indices]
    scores_b = scores[indices]
    pred_b = y_pred[indices]

    bootstrap_results["accuracy"].append(
        accuracy_score(y_b, pred_b)
    )
    bootstrap_results["precision"].append(
        precision_score(y_b, pred_b, zero_division=0)
    )
    bootstrap_results["recall"].append(
        recall_score(y_b, pred_b, zero_division=0)
    )
    bootstrap_results["f1"].append(
        f1_score(y_b, pred_b, zero_division=0)
    )

    # AUC metrics require both classes to be present in the sample.
    if len(np.unique(y_b)) == 2:
        bootstrap_results["roc_auc"].append(
            roc_auc_score(y_b, scores_b)
        )
        bootstrap_results["pr_auc"].append(
            average_precision_score(y_b, scores_b)
        )
    else:
        bootstrap_results["roc_auc"].append(np.nan)
        bootstrap_results["pr_auc"].append(np.nan)

# Calculate percentile-based 95% confidence intervals.
ci_rows = []

point_estimates = {
    "accuracy": accuracy_score(y_true, y_pred),
    "precision": precision_score(y_true, y_pred, zero_division=0),
    "recall": recall_score(y_true, y_pred, zero_division=0),
    "f1": f1_score(y_true, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_true, scores),
    "pr_auc": average_precision_score(y_true, scores)
}

for metric in metric_names:
    values = np.array(bootstrap_results[metric], dtype=float)
    lower, upper = np.nanpercentile(values, [2.5, 97.5])

    ci_rows.append({
        "metric": metric,
        "test_value": point_estimates[metric],
        "ci_95_lower": lower,
        "ci_95_upper": upper
    })

nli_bootstrap_ci = pd.DataFrame(ci_rows)

print("NLI test metrics with 95% bootstrap confidence intervals:")
display(nli_bootstrap_ci.round(4))

# Save results to Google Drive.
output_path = (
    "/content/drive/MyDrive/rag-eval-metrics/"
    "nli_canonical_test_bootstrap_ci.csv"
)

nli_bootstrap_ci.to_csv(output_path, index=False)
print("\nSaved to:", output_path)

NLI test metrics with 95% bootstrap confidence intervals:


,metric,test_value,ci_95_lower,ci_95_upper
0,accuracy,0.4762,0.4258,0.5266
1,precision,0.4472,0.3963,0.5000
2,recall,0.9412,0.9028,0.9747
3,f1,0.6063,0.5563,0.6545
4,roc_auc,0.5115,0.4629,0.5585
5,pr_auc,0.4313,0.3748,0.4892



Saved to: /content/drive/MyDrive/rag-eval-metrics/nli_canonical_test_bootstrap_ci.csv


In [ ]:
#Save the NLI scores and test results

import os

OUTPUT_DIR = "/content/drive/MyDrive/rag-eval-metrics"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save response-level scores with ground-truth labels and split assignments.
response_scores_path = os.path.join(
    OUTPUT_DIR, "ragtruth_nli_response_scores.csv"
)
nli_eval_df.to_csv(response_scores_path, index=False)

# Save sentence-level predictions and probabilities.
sentence_scores_path = os.path.join(
    OUTPUT_DIR, "ragtruth_nli_sentence_scores.csv"
)
nli_pairs_df.to_csv(sentence_scores_path, index=False)

# Save the test metrics and bootstrap confidence intervals.
metrics_path = os.path.join(
    OUTPUT_DIR, "nli_canonical_test_bootstrap_ci.csv"
)
nli_bootstrap_ci.to_csv(metrics_path, index=False)

print("Saved NLI outputs:")
print(response_scores_path)
print(sentence_scores_path)
print(metrics_path)

Saved NLI outputs:
/content/drive/MyDrive/rag-eval-metrics/ragtruth_nli_response_scores.csv
/content/drive/MyDrive/rag-eval-metrics/ragtruth_nli_sentence_scores.csv
/content/drive/MyDrive/rag-eval-metrics/nli_canonical_test_bootstrap_ci.csv


In [ ]:
#Check the metric metadata file

import os
import pandas as pd

metadata_path = (
    "/content/drive/MyDrive/rag-eval-metrics/metric_run_metadata.csv"
)

if os.path.exists(metadata_path):
    metric_metadata = pd.read_csv(metadata_path)

    print("Metadata file found.")
    print("Shape:", metric_metadata.shape)
    print("\nColumns:")
    print(metric_metadata.columns.tolist())
    print("\nExisting metric records:")
    display(metric_metadata)
else:
    print("Metadata file not found at:")
    print(metadata_path)
    print("\nWe can create it with the completed metric records.")

Metadata file found.
Shape: (2, 12)

Columns:
['metric', 'model_or_method', 'device', 'compute_time_seconds', 'val_threshold', 'val_f1', 'test_accuracy', 'test_precision', 'test_recall', 'test_f1', 'test_roc_auc', 'test_pr_auc']

Existing metric records:


,metric,model_or_method,device,compute_time_seconds,val_threshold,val_f1,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc
0,rouge1_precision,rouge-score (ROUGE-1),cpu,NaN,0.811800,0.7016,0.6527,0.5622,0.8562,0.6788,0.7298,0.6060
1,bertscore_precision,roberta-large,cuda (T4),169.47,0.902768,0.6815,0.5770,0.5036,0.9020,0.6464,0.7219,0.6095


In [ ]:
import pandas as pd
import os

metadata_path = (
    "/content/drive/MyDrive/rag-eval-metrics/metric_run_metadata.csv"
)

# Load existing metadata
metric_metadata = pd.read_csv(metadata_path)

# NLI results
nli_record = {
    "metric": "nli_hallucination_score (1 - entailment_fraction)",
    "model_or_method": "roberta-large-mnli",
    "device": "cuda (T4)",
    "compute_time_seconds": 1305,
    "val_threshold": 0.600000,
    "val_f1": 0.6128,
    "test_accuracy": 0.4762,
    "test_precision": 0.4472,
    "test_recall": 0.9412,
    "test_f1": 0.6063,
    "test_roc_auc": 0.5115,
    "test_pr_auc": 0.4313
}

# Remove any previous NLI record so rerunning this cell won't duplicate it
metric_metadata = metric_metadata[
    metric_metadata["metric"] != nli_record["metric"]
]

# Append the NLI record
metric_metadata = pd.concat(
    [metric_metadata, pd.DataFrame([nli_record])],
    ignore_index=True
)

# Save updated metadata
metric_metadata.to_csv(metadata_path, index=False)

print("Updated metadata saved successfully.")
print("Shape:", metric_metadata.shape)
display(metric_metadata)

Updated metadata saved successfully.
Shape: (3, 12)


,metric,model_or_method,device,compute_time_seconds,val_threshold,val_f1,test_accuracy,test_precision,test_recall,test_f1,test_roc_auc,test_pr_auc
0,rouge1_precision,rouge-score (ROUGE-1),cpu,NaN,0.811800,0.7016,0.6527,0.5622,0.8562,0.6788,0.7298,0.6060
1,bertscore_precision,roberta-large,cuda (T4),169.47,0.902768,0.6815,0.5770,0.5036,0.9020,0.6464,0.7219,0.6095
2,nli_hallucination_score (1 - entailment_fraction),roberta-large-mnli,cuda (T4),1305.00,0.600000,0.6128,0.4762,0.4472,0.9412,0.6063,0.5115,0.4313
